# Sid — Model Evaluation

Comprehensive evaluation and comparison of all models built
across classification and regression tasks.

**Input:** `../data/processed/model_ready_dataset.csv`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score,
                             roc_curve, mean_squared_error,
                             mean_absolute_error, r2_score)
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

### (2) Load and prepare data

In [2]:
df = pd.read_csv(
    "../data/processed/model_ready_dataset.csv",
    dtype={"county_fips": str},
    low_memory=False
)

features = [
    "higher_ed_rate", "poverty_rate", "unemployment_rate",
    "white_pct", "black_pct", "asian_pct",
    "median_income", "log_population", "housing_density"
]

model_df = df[features + ["party_winner", "dem_vote_share"]].dropna()

X = model_df[features]
y_class = model_df["party_winner"]
y_reg   = model_df["dem_vote_share"]

X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_class, y_reg,
    test_size=0.2, random_state=42, stratify=y_class
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (2491, 9) | Test: (623, 9)


### (3) Train all models

In [3]:
# Classification models
lr_clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
rf_clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
xgb_clf = XGBClassifier(n_estimators=200, scale_pos_weight=(yc_train==0).sum()/(yc_train==1).sum(),
                         random_state=42, eval_metric="logloss", verbosity=0)

lr_clf.fit(X_train_scaled, yc_train)
rf_clf.fit(X_train_scaled, yc_train)
xgb_clf.fit(X_train_scaled, yc_train)

# Regression models
lr_reg    = LinearRegression()
ridge_reg = Ridge(alpha=1.0)
lasso_reg = Lasso(alpha=0.001)
rf_reg    = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

lr_reg.fit(X_train_scaled, yr_train)
ridge_reg.fit(X_train_scaled, yr_train)
lasso_reg.fit(X_train_scaled, yr_train)
rf_reg.fit(X_train_scaled, yr_train)

print("All models trained.")

All models trained.


### (4) Cross-validation — classification

In [4]:
print("5-Fold Cross-Validation (F1 score — Democrat class):")
print("=" * 50)

for name, model in [
    ("Logistic Regression", lr_clf),
    ("Random Forest",       rf_clf),
    ("XGBoost",             xgb_clf),
]:
    scores = cross_val_score(
        model, X_train_scaled, yc_train,
        cv=5, scoring="f1", n_jobs=-1
    )
    print(f"{name}")
    print(f"  Scores: {scores.round(3)}")
    print(f"  Mean: {scores.mean():.3f} | Std: {scores.std():.3f}")
    print()

5-Fold Cross-Validation (F1 score — Democrat class):
Logistic Regression
  Scores: [0.713 0.696 0.694 0.647 0.685]
  Mean: 0.687 | Std: 0.022

Random Forest
  Scores: [0.777 0.685 0.78  0.731 0.774]
  Mean: 0.749 | Std: 0.037

XGBoost
  Scores: [0.807 0.703 0.789 0.738 0.751]
  Mean: 0.758 | Std: 0.037



### (5) Cross-validation — regression

In [5]:
print("5-Fold Cross-Validation (R² score):")
print("=" * 50)

for name, model in [
    ("Linear Regression", lr_reg),
    ("Ridge",             ridge_reg),
    ("Lasso",             lasso_reg),
    ("Random Forest",     rf_reg),
]:
    scores = cross_val_score(
        model, X_train_scaled, yr_train,
        cv=5, scoring="r2", n_jobs=-1
    )
    print(f"{name}")
    print(f"  Scores: {scores.round(3)}")
    print(f"  Mean: {scores.mean():.3f} | Std: {scores.std():.3f}")
    print()

5-Fold Cross-Validation (R² score):
Linear Regression
  Scores: [0.669 0.66  0.694 0.657 0.683]
  Mean: 0.673 | Std: 0.014

Ridge
  Scores: [0.669 0.66  0.694 0.657 0.683]
  Mean: 0.673 | Std: 0.014

Lasso
  Scores: [0.669 0.66  0.695 0.656 0.682]
  Mean: 0.672 | Std: 0.014

Random Forest
  Scores: [0.75  0.744 0.754 0.72  0.745]
  Mean: 0.743 | Std: 0.012



### (6) Final test set evaluation — classification

In [6]:
clf_results = []

for name, model in [
    ("Logistic Regression", lr_clf),
    ("Random Forest",       rf_clf),
    ("XGBoost",             xgb_clf),
]:
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    report = classification_report(yc_test, y_pred,
                                   target_names=["Republican","Democrat"],
                                   output_dict=True)
    clf_results.append({
        "Model":               name,
        "Accuracy":            round(report["accuracy"], 3),
        "Democrat F1":         round(report["Democrat"]["f1-score"], 3),
        "Democrat Precision":  round(report["Democrat"]["precision"], 3),
        "Democrat Recall":     round(report["Democrat"]["recall"], 3),
        "ROC AUC":             round(roc_auc_score(yc_test, y_prob), 3),
        "CV F1 Mean":          round([0.687, 0.749, 0.758][["Logistic Regression","Random Forest","XGBoost"].index(name)], 3)
    })

clf_df = pd.DataFrame(clf_results).set_index("Model")
print("CLASSIFICATION RESULTS")
print("=" * 60)
display(clf_df)

CLASSIFICATION RESULTS


,Accuracy,Democrat F1,Democrat Precision,Democrat Recall,ROC AUC,CV F1 Mean
Model,,,,,,
Logistic Regression,0.883,0.727,0.610,0.898,0.952,0.687
Random Forest,0.929,0.788,0.820,0.759,0.960,0.749
XGBoost,0.917,0.768,0.741,0.796,0.961,0.758


### (7) Final test set evaluation — regression

In [7]:
reg_results = []

for name, model in [
    ("Linear Regression", lr_reg),
    ("Ridge",             ridge_reg),
    ("Lasso",             lasso_reg),
    ("Random Forest",     rf_reg),
]:
    y_pred = model.predict(X_test_scaled)
    reg_results.append({
        "Model": name,
        "R²":    round(r2_score(yr_test, y_pred), 4),
        "RMSE":  round(np.sqrt(mean_squared_error(yr_test, y_pred)), 4),
        "MAE":   round(mean_absolute_error(yr_test, y_pred), 4),
        "CV R² Mean": round([0.673, 0.673, 0.672, 0.743][
            ["Linear Regression","Ridge","Lasso","Random Forest"].index(name)], 3)
    })

reg_df = pd.DataFrame(reg_results).set_index("Model")
print("REGRESSION RESULTS")
print("=" * 60)
display(reg_df)

REGRESSION RESULTS


,R²,RMSE,MAE,CV R² Mean
Model,,,,
Linear Regression,0.6688,0.0943,0.0739,0.673
Ridge,0.6688,0.0943,0.0738,0.673
Lasso,0.6708,0.0940,0.0736,0.672
Random Forest,0.7367,0.0841,0.0629,0.743


### (8) Final summary

In [8]:
print("=" * 55)
print("FULL PROJECT MODEL EVALUATION SUMMARY")
print("=" * 55)

print("\nCLASSIFICATION (predicting party_winner):")
print("-" * 55)
print("Best model:        Random Forest")
print("Accuracy:          0.929")
print("Democrat F1:       0.788")
print("Democrat Precision:0.820")
print("Democrat Recall:   0.759")
print("ROC AUC:           0.960")
print("CV F1 Mean:        0.749")

print("\nREGRESSION (predicting dem_vote_share):")
print("-" * 55)
print("Best model:        Random Forest Regressor")
print("R² Score:          0.737")
print("RMSE:              0.084")
print("MAE:               0.063")
print("CV R² Mean:        0.743")

print("\nCLUSTERING (county grouping):")
print("-" * 55)
print("Algorithm:         K-Means (k=3)")
print("Silhouette Score:  0.309")
print("Clusters found:")
print("  Cluster 0 — Urban/Diverse   (avg dem share: 0.50)")
print("  Cluster 1 — Mixed/Swing     (avg dem share: 0.42)")
print("  Cluster 2 — Rural/White     (avg dem share: 0.26)")

print("\nKEY FINDINGS:")
print("-" * 55)
print("1. Random Forest best model for both tasks")
print("2. Top features: white_pct, higher_ed_rate, asian_pct")
print("3. Class imbalance (4.78x) handled via class_weight")
print("4. Demographics alone explain 74% of vote share variance")
print("5. K-Means found political groups without election labels")
print("=" * 55)

FULL PROJECT MODEL EVALUATION SUMMARY

CLASSIFICATION (predicting party_winner):
-------------------------------------------------------
Best model:        Random Forest
Accuracy:          0.929
Democrat F1:       0.788
Democrat Precision:0.820
Democrat Recall:   0.759
ROC AUC:           0.960
CV F1 Mean:        0.749

REGRESSION (predicting dem_vote_share):
-------------------------------------------------------
Best model:        Random Forest Regressor
R² Score:          0.737
RMSE:              0.084
MAE:               0.063
CV R² Mean:        0.743

CLUSTERING (county grouping):
-------------------------------------------------------
Algorithm:         K-Means (k=3)
Silhouette Score:  0.309
Clusters found:
  Cluster 0 — Urban/Diverse   (avg dem share: 0.50)
  Cluster 1 — Mixed/Swing     (avg dem share: 0.42)
  Cluster 2 — Rural/White     (avg dem share: 0.26)

KEY FINDINGS:
-------------------------------------------------------
1. Random Forest best model for both tasks
2. Top fe